# Hydroforming Operations Analysis

## A focused review of machine continuity, interruption, throughput, and scrap

This notebook presents the main findings from the three-month SCAMP report in a shorter and more structured form. The analysis keeps the original selected code unchanged and adds clearer explanations around the results.

The main questions are:

- What does one row, job, product, and machine represent in this dataset?
- How continuous is production, and how are gaps related to job changes?
- Which product and machine combinations lose the most time to interruptions?
- Does the same product perform differently across machines?
- Which machines and weeks show the highest interruption and scrap rates?
- Are interruptions different around job transitions?

The purpose is descriptive: to identify where further operational investigation should begin, not to claim a cause from the available data alone.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import sqlite3 as sql1
import plotly.express as px

In [2]:
conn=sql1.connect("data/hydroforming-scamp-db-sqlite.db")

def run_query(query,conn):
    df=pd.read_sql(query,conn)
    return df

## 2. Dataset structure and grain

The first step is to confirm the level of detail in the report and the relationships between its main IDs. This prevents the later KPIs from being grouped at the wrong level.

In [3]:
query=""" 
SELECT 
    COUNT(*) AS TotalRow,
    COUNT(DISTINCT(ReportID)) AS DistinctReportID,
    COUNT(DISTINCT(JobID)) AS DistinctJobID,
    COUNT(DISTINCT(DeviceID)) AS DistinctDeviceID,
    COUNT(DISTINCT(ProductID)) AS DistinctJProductID

FROM scamp_report_3m
"""


df=run_query(query,conn)
df.head()

,TotalRow,DistinctReportID,DistinctJobID,DistinctDeviceID,DistinctJProductID
0,2793,2793,153,5,30


In [4]:
query="""  
    SELECT 
        MIN(strftime('%Y-%m', StartTime) )AS StartingPeriod,
        MAX(strftime('%Y-%m', StartTime) )AS EndingPeriod
    FROM scamp_report_3m
"""

df=run_query(query,conn)
df.head()

,StartingPeriod,EndingPeriod
0,2023-06,2023-09


In [5]:
query=""" 
WITH ct AS
(
SELECT 
    JobID,
    COUNT(DISTINCT(ProductID)) AS ProductIDsCountGroupedByJobID,
    COUNT(DISTINCT(DeviceID)) AS DeviceIDsCountGroupedByJobID
    

FROM scamp_report_3m
GROUP BY JobID

)

SELECT 
COUNT(JobID) AS TotalUniqueJobID,
SUM(ProductIDsCountGroupedByJobID) AS UniqueProductIDsAgainstJobIDs,
SUM(DeviceIDsCountGroupedByJobID) AS UniqueDeviceIDsAgainstJobIDs
FROM ct
"""


# query=""" 

# SELECT 
#     JobID,
#     COUNT(DISTINCT(ProductID)) AS ProductIDsCountGroupedByJobID
    

# FROM scamp_report_3m
# GROUP BY JobID


# """

df=run_query(query,conn)
print(df.head(10))

   TotalUniqueJobID  UniqueProductIDsAgainstJobIDs  \
0               153                            153   

   UniqueDeviceIDsAgainstJobIDs  
0                           153  


### Summary

- The table contains **2,793 rows and 2,793 unique ReportIDs**, so one row can be treated as one production report.
- The data contains **153 jobs, 30 products, and 5 machines**, covering **June 2023 to early September 2023**.
- Every JobID is associated with one ProductID and one DeviceID. A job therefore represents one product running on one machine, although the same product can appear in other jobs and on other machines.
- The first and last calendar weeks are partial weeks, so their weekly rates should be compared with care.

Machine reference: Device 168 = Hydroforming 9, 169 = Hydroforming 5, 170 = Hydroforming 8, 172 = Hydroforming 4, and 321 = Hydroforming 12.

## 3. Production continuity and gaps

A gap is measured as the time between the end of one report and the start of the next report on the same machine. This section checks whether reports overlap, how long the common gaps are, and whether gaps usually occur with a job change.

In [6]:
query=""" 
WITH t AS (
    SELECT
        ReportID,
        DeviceID,
        StartTime,
        EndTime,

        LAG(EndTime) OVER (
            PARTITION BY DeviceID
            ORDER BY StartTime
        ) AS PreviousReportEndTime

    FROM scamp_report_3m
),

t2 AS (
SELECT *,
       ROUND(
           (julianday(StartTime) - julianday(PreviousReportEndTime))
           * 24 * 60,
           2
       ) AS GapMinutes

FROM t

ORDER BY DeviceID, StartTime

),

t3 AS
(
SELECT *,
    CASE 
    WHEN GapMinutes < 0 THEN 'overlap'
    WHEN GapMinutes = 0 THEN 'continuous' 
    ELSE 'gap'
    END AS Transitions
FROM t2
)

SELECT DeviceID,Transitions,COUNT(Transitions)
FROM t3
GROUP BY DeviceID, Transitions



"""


df=run_query(query,conn)
print(df.head(20))
print("\n")
print(df["Transitions"].value_counts())

   DeviceID Transitions  COUNT(Transitions)
0       168  continuous                 438
1       168         gap                 152
2       169  continuous                 379
3       169         gap                 149
4       170  continuous                 462
5       170         gap                 158
6       172  continuous                 285
7       172         gap                 133
8       321  continuous                 470
9       321         gap                 167


Transitions
continuous    5
gap           5
Name: count, dtype: int64


There are no negative gaps, so no report overlap is found within a machine timeline. One technical point is that the query above places the first record of each machine in the `gap` category because its previous end time is null. Excluding those five first records leaves **754 positive gaps**.

In [7]:
query=""" 
WITH t AS (
    SELECT
        ReportID,
        DeviceID,
        StartTime,
        EndTime,

        LAG(EndTime) OVER (
            PARTITION BY DeviceID
            ORDER BY StartTime
        ) AS PreviousReportEndTime

    FROM scamp_report_3m
),

t2 AS (
SELECT *,
       ROUND(
           (julianday(StartTime) - julianday(PreviousReportEndTime))
           * 24 * 60,
           2
       ) AS GapMinutes

FROM t

ORDER BY DeviceID, StartTime

),

t3 AS
(
SELECT *,
    CASE 
    WHEN GapMinutes < 0 THEN 'overlap'
    WHEN GapMinutes = 0 THEN 'continuous' 
    ELSE 'gap'
    END AS Transitions
FROM t2
)

SELECT GapMinutes
FROM t3
WHERE GapMinutes > 0 AND GapMinutes < 40



"""


df=run_query(query,conn)
print(df.head(10))
print(df.median())
print(df.describe())


fig=px.histogram(
    df,
    x="GapMinutes",
    nbins=12,
    text_auto=True,
    title="Distribution of Production Gaps Under 40 Minutes",
    labels={"GapMinutes": "Gap Duration (minutes)"}
)

fig.update_xaxes(title_text="Gap Duration (minutes)", automargin=True)
fig.update_yaxes(title_text="Report Count", automargin=True)



fig.show()
# print(df["Transitions"].value_counts())

   GapMinutes
0        5.68
1        7.37
2        7.37
3       12.63
4        5.45
5        7.37
6        8.08
7        8.67
8        7.65
9        7.37
GapMinutes    8.24
dtype: float64
       GapMinutes
count  568.000000
mean     8.796884
std      3.923658
min      1.350000
25%      7.370000
50%      8.240000
75%      9.230000
max     36.600000


For the **568 gaps below 40 minutes**, the median is **8.24 minutes** and the middle 50% lie between **7.37 and 9.23 minutes**. This shows a clear group of short, routine gaps. The longer gaps are less frequent but should be reviewed separately because they can represent off-shift time, planned stops, missing reporting time, or longer downtime events.

In [8]:
query=""" 
WITH t AS (
    SELECT
        ReportID,
        DeviceID,
        StartTime,
        EndTime,
        JobID,


        LAG(EndTime) OVER (
            PARTITION BY DeviceID
            ORDER BY StartTime
        ) AS PreviousReportEndTime,

        LAG(jobID) OVER (
            PARTITION BY DeviceID
            ORDER BY StartTime
                ) AS PreviousJobID


    FROM scamp_report_3m
),

t2 AS (
SELECT *,
       ROUND(
           (julianday(StartTime) - julianday(PreviousReportEndTime))
           * 24 * 60,
           2
       ) AS GapMinutes

FROM t



),

t3 AS
(
SELECT *,
    CASE 
        WHEN GapMinutes IS NULL THEN 'first_record'
        WHEN GapMinutes < 0 THEN 'overlap'
        WHEN GapMinutes = 0 THEN 'continuous'
        ELSE 'gap'
    END AS Transitions
FROM t2
),

t4 AS
(
SELECT *,
    CASE
    WHEN PreviousJobID IS NULL THEN 'first_record'
    WHEN JobID <> PreviousJobID THEN 'YES'
    ELSE 'NO'
    END AS DidJobChanged
FROM t3
)

SELECT DidJobChanged,Transitions,COUNT(Transitions)
FROM t4
GROUP BY DidJobChanged,Transitions



"""


df=run_query(query,conn)
print(df.head(10))

  DidJobChanged   Transitions  COUNT(Transitions)
0            NO    continuous                1961
1            NO           gap                 673
2           YES    continuous                  73
3           YES           gap                  81
4  first_record  first_record                   5


### Summary

- Of 754 positive gaps, **673 occurred without a job change** and **81 occurred when the job changed**.
- Therefore, only about **10.7% of gaps** are directly paired with a job change in the report sequence.
- Looking from the other direction, **81 of 154 job changes (52.6%)** have a positive gap, while 73 are recorded as continuous.

This means a gap should not automatically be treated as setup or changeover time. Most gaps occur within the same job, while roughly half of job changes show no gap between report timestamps.

## 4. Interruption impact by product and machine

The next comparison estimates how much reported time is lost to recorded interruptions for each ProductID and DeviceID combination. The percentage is based on interruption time divided by total reported time.

In [9]:
query= """  
WITH tp AS
(SELECT 
    DeviceID,
    ProductID,
    SUM((julianday(EndTime) - julianday(StartTime))*24) AS ProductionTimeInHRS,
    SUM(QuantityProduced) AS TotalProduced,
    SUM(((julianday(EndTime) - julianday(StartTime))*24)-(InteruptMinutes/60)) AS ProductionTimeInHRSExcIT


FROM scamp_report_3m
GROUP BY DeviceID,ProductID
),

tp2 AS
(
SELECT *,(TotalProduced/ProductionTimeInHRS) AS RunTimeThroughPut,(TotalProduced/ProductionTimeInHRSExcIT) AS ObservedThroughPut
FROM tp
ORDER BY RunTimeThroughPut DESC
)

SELECT DeviceID,ProductID,ProductionTimeInHRS,((ObservedThroughPut-RunTimeThroughPut)/ObservedThroughPut)*100 AS PercImpactOfInterruptionTime 
FROM tp2



"""

df=run_query(query,conn)
print(df.head())

df["DeviceID"]=df["DeviceID"].astype(str)
df["ProductID"]=df["ProductID"].astype(str)

fig=px.box(
    df,
    x="PercImpactOfInterruptionTime",
    y="ProductID",
    # hover_data=["PercImpactOfInterruptionTime"]
    hover_data=["ProductionTimeInHRS"],
    title="Interruption Impact on Throughput by Product",
    labels={
        "PercImpactOfInterruptionTime": "Interruption Impact (%)",
        "ProductID": "Product"
    }
)



fig.update_layout(width=800, height=2000)
fig.update_xaxes(ticksuffix="%", automargin=True)
fig.update_yaxes(automargin=True)

fig.show()



   DeviceID  ProductID  ProductionTimeInHRS  PercImpactOfInterruptionTime
0       169        302            22.332778                     15.665563
1       169        443             2.031667                      0.000000
2       172        302           145.676667                     18.917414
3       169        249             2.542500                      7.942751
4       172        495             5.612222                     28.766581


The box plot gives the distribution by product, while the heatmap below makes the product–machine combinations easier to compare.

In [10]:
heatmap_df = df.pivot(
    index="ProductID",
    columns="DeviceID",
    values="PercImpactOfInterruptionTime"
)

product_order = (
    heatmap_df.max(axis=1, skipna=True)
    .sort_values(ascending=False, kind="stable", na_position="last")
    .index
)
heatmap_df = heatmap_df.loc[product_order]

fig = px.imshow(
    heatmap_df,
    text_auto=".1f",
    aspect="auto",
    title="Interruption Impact by Product and Machine",
    color_continuous_scale="Mint",
    labels={
        "x": "Machine",
        "y": "Product",
        "color": "Interruption impact (%)"
    }
)
fig.update_layout(
    height=1000
)
fig.update_xaxes(automargin=True)
fig.update_yaxes(automargin=True)
fig.show()

### Summary

- Some of the highest percentages belong to combinations with only a few reported hours. For example, Product 226 on Device 170 shows a very high interruption percentage but has only about 3.8 reported hours and no recorded output. These cases are important to inspect, but they should not be ranked without an exposure threshold.
- Among combinations with substantial reported time, Product 302 on Device 172 loses about **18.9%** of reported time across roughly **145.7 hours**. Product 370 on Device 169 loses about **15.7%** across **125.2 hours**, and Product 322 on Device 169 loses about **13.3%** across **132.3 hours**.
- The pattern is not uniform across machines, which suggests that product requirements and machine conditions should be investigated together.

## 5. Product × machine throughput

To check whether the same product performs differently depending on the machine, throughput is first calculated at ProductID × DeviceID × Week level. Observed throughput includes all reported time; runtime throughput removes the recorded interruption minutes from that time.

In [11]:
query=""" 
WITH t1 AS
(
SELECT ProductID,DeviceID,strftime('%Y-%W',StartTime) AS WeekOfYear,SUM(QuantityProduced) AS QuantityProduced,SUM((julianday(EndTime)-julianday(StartTime))*24) AS TotalReportedHours, SUM(((julianday(EndTime)-julianday(StartTime))*24)-(InteruptMinutes/60.0)) AS EstimatedRunHours
FROM scamp_report_3m
GROUP BY ProductID,DeviceID,strftime('%Y-%W',StartTime)
),

t2 AS
(
SELECT *, (QuantityProduced/TotalReportedHours) AS ObservedThroughput, (QuantityProduced/EstimatedRunHours) AS RunTimeThroughput
FROM t1

)

SELECT *,(RunTimeThroughput - ObservedThroughput)*100/RunTimeThroughput AS EstimatedThroughputLossPct
FROM t2



"""

df=run_query(query,conn)
df.head(10)


,ProductID,DeviceID,WeekOfYear,QuantityProduced,TotalReportedHours,EstimatedRunHours,ObservedThroughput,RunTimeThroughput,EstimatedThroughputLossPct
0,189,169,2023-23,3533.0,24.048333,23.458113,146.912468,150.608874,2.454308
1,189,169,2023-28,3544.0,23.791389,21.210373,148.961459,167.088053,10.848528
2,189,170,2023-30,2074.0,14.241111,14.048889,145.634704,147.627333,1.349770
3,189,170,2023-31,3206.0,18.905833,18.555556,169.577291,172.778443,1.852750
4,189,172,2023-31,2376.0,14.674167,14.357500,161.917201,165.488420,2.157987
5,212,168,2023-23,1810.0,15.286667,12.728581,118.403838,142.199666,16.734096
6,212,168,2023-24,1128.0,8.842778,8.282500,127.561726,136.190763,6.335993
7,212,168,2023-30,1642.0,13.789444,13.403611,119.076588,122.504300,2.798034
8,226,170,2023-36,0.0,3.811667,0.444811,0.000000,0.000000,NaN
9,227,168,2023-28,1986.0,20.459722,16.180967,97.068766,122.736795,20.913067


In [12]:
df["ProductID"]=df["ProductID"].astype(str)
df["DeviceID"]=df["DeviceID"].astype(str)

a = (
    df.groupby(["ProductID", "DeviceID"], as_index=False)
      .agg(
          QuantityProduced=("QuantityProduced", "sum"),
          TotalReportedHours=("TotalReportedHours", "sum")
      )
)

print(a)

a["ObservedThroughput"] = (
    a["QuantityProduced"] / a["TotalReportedHours"]
)

fig=px.density_heatmap(
    a,
    y="ProductID",
    x="DeviceID",
    z="ObservedThroughput",
    color_continuous_scale="Mint",
    title="Observed Throughput by Product and Machine",
    labels={
        "DeviceID": "Machine",
        "ProductID": "Product",
        "ObservedThroughput": "Observed Throughput"
    },
    

)

fig.update_yaxes(
    showgrid=True,             # Turn Y-axis grid lines ON (set to False to turn OFF)
    gridwidth=2,               # Change width of the grid lines
    automargin=True,
)

fig.update_xaxes(automargin=True)

fig.update_layout(height=1000)

fig.show()

   ProductID DeviceID  QuantityProduced  TotalReportedHours
0        189      169            7077.0           47.839722
1        189      170            5280.0           33.146944
2        189      172            2376.0           14.674167
3        212      168            4580.0           37.918889
4        226      170               0.0            3.811667
5        227      168            1986.0           20.459722
6        227      169            3463.0           31.870000
7        227      170           14396.0          107.187500
8        227      172           16972.0          134.595556
9        232      170             654.0            5.582778
10       238      170             578.0            5.384167
11       244      169            2784.0           19.056944
12       249      169             434.0            2.542500
13       265      169            6515.0           44.645000
14       265      172            1434.0           10.301389
15       290      169            1469.0 

### Summary

- The heatmap shows visible throughput differences for several products that ran on more than one machine.
- Product 507 has the largest descriptive spread, from about **76.5 to 127.9 units per reported hour** across two machines. Product 401 ranges from about **80.0 to 112.7**, and Product 227 from about **97.1 to 134.3**.
- Product 297 has a smaller but still meaningful range of about **122.4 to 144.2 units per reported hour**, supported by comparatively high production volume on four machines.

These are useful root-cause directions, but not controlled machine comparisons. Product mix by week, run length, interruption, staffing, material, and operating settings can also affect the observed rate.

## 6. Weekly machine performance

Weekly interruption and scrap rates are used to locate periods of weaker performance. Aggregating the numerator and denominator before calculating each rate keeps the result weighted by actual machine activity.

In [13]:
query = """ 

WITH t1 AS
(
SELECT DeviceID,strftime('%Y-%W',StartTime) AS WeekOfYear,SUM(QuantityProduced) AS QuantityProduced, SUM(QuantityScrap) AS QuantityScrap,SUM((julianday(EndTime)-julianday(StartTime))*24) AS TotalReportedHours, SUM(((julianday(EndTime)-julianday(StartTime))*24)-(InteruptMinutes/60.0)) AS EstimatedRunHours, SUM(InteruptMinutes/60.0) AS InterruptionHours
FROM scamp_report_3m
GROUP BY DeviceID,strftime('%Y-%W',StartTime)
)

SELECT *,(TotalReportedHours-EstimatedRunHours)*100/TotalReportedHours AS InterruptionRate, QuantityScrap/QuantityProduced*100 AS ScrapRate
FROM t1

"""

df=run_query(query,conn)
print(df.head())


df["DeviceID"]=df["DeviceID"].astype(str)


   DeviceID WeekOfYear  QuantityProduced  QuantityScrap  TotalReportedHours  \
0       168    2023-22             514.0            2.0            3.860556   
1       168    2023-23            7873.0           60.0           84.793889   
2       168    2023-24           12462.0          261.0          107.217778   
3       168    2023-25           13008.0           61.0          125.116389   
4       168    2023-26           15892.0          133.0          132.978889   

   EstimatedRunHours  InterruptionHours  InterruptionRate  ScrapRate  
0           3.789722           0.070833          1.834796   0.389105  
1          73.557444          11.236445         13.251479   0.762098  
2          98.929444           8.288333          7.730372   2.094367  
3         108.735350          16.381039         13.092641   0.468942  
4         121.419922          11.558967          8.692332   0.836899  


In [14]:
df.head()


c=df.groupby(["WeekOfYear","DeviceID"], as_index=False).agg(
    QuantityProduced=("QuantityProduced","sum"),
    QuantityScrap=("QuantityScrap","sum"),
)

c["ScrapRate"]=c["QuantityScrap"]/c["QuantityProduced"]*100
c.head(10)


fig=px.line(
    c,
    x="WeekOfYear",
    y="ScrapRate",
    color="DeviceID",
    title="Weekly Scrap Rate by Machine",
    labels={
        "WeekOfYear": "Week",
        "ScrapRate": "Scrap Rate (%)",
        "DeviceID": "Machine"
    }
)

fig.update_xaxes(tickangle=-45, automargin=True)
fig.update_yaxes(ticksuffix="%", automargin=True)

fig.show()

In [15]:
df.head()

d=df.groupby(["WeekOfYear","DeviceID"]).agg(
    InterruptionHours=("InterruptionHours","sum"),
    TotalReportedHours=("TotalReportedHours","sum")
                       
).reset_index()
d["InterruptionRate"]=d["InterruptionHours"] / d["TotalReportedHours"] *100

fig=px.line(
    d,
    x="WeekOfYear",
    y="InterruptionRate",
    color="DeviceID",
    title="Weekly Interruption Rate by Machine",
    labels={
        "WeekOfYear": "Week",
        "InterruptionRate": "Interruption Rate (%)",
        "DeviceID": "Machine"
    }
)

fig.update_xaxes(tickangle=-45, automargin=True)
fig.update_yaxes(ticksuffix="%", automargin=True)

fig.show()

### Summary

- Across the full period, Device 169 has the highest interruption rate at about **11.1%** and the highest scrap rate at about **2.85%**.
- Devices 170 and 321 have the lowest overall interruption rates, at about **7.65%** and **7.08%** respectively.
- Device 169 shows a clear scrap problem during weeks 26–29. Its weekly scrap rate reaches about **8.64% in week 29**.
- Some large interruption peaks occur in partial weeks, including Device 170 in week 36. These edge weeks should not be used alone for performance ranking because their recorded hours are incomplete.

The weekly view points most strongly toward Device 169 for scrap investigation, while interruption performance needs both machine-level and week-level follow-up.

## 7. Scrap investigation: Device 169 and Product 297

Device 169 is isolated first to identify which product is behind its elevated scrap rate. Product 297 stands out clearly, so its weekly behavior and performance across other machines are then compared.

In [16]:


query=""" 
WITH t1 AS
(
SELECT ProductID,DeviceID,strftime('%Y-%W',StartTime) AS WeekOfYear,SUM(QuantityScrap) AS QuantityScrap,SUM(QuantityProduced) AS QuantityProduced,SUM((julianday(EndTime)-julianday(StartTime))*24) AS TotalReportedHours, SUM(((julianday(EndTime)-julianday(StartTime))*24)-(InteruptMinutes/60.0)) AS EstimatedRunHours
FROM scamp_report_3m
GROUP BY ProductID,DeviceID,strftime('%Y-%W',StartTime)
),

t2 AS
(
SELECT *, (QuantityProduced/TotalReportedHours) AS ObservedThroughput, (QuantityProduced/EstimatedRunHours) AS RunTimeThroughput
FROM t1

)

SELECT *,(RunTimeThroughput - ObservedThroughput)*100/RunTimeThroughput AS EstimatedThroughputLossPct
FROM t2



"""

df=run_query(query,conn)
df.head(10)

df["DeviceID"]=df["DeviceID"].astype(str)
df["ProductID"]=df["ProductID"].astype(str)

df=df[df["DeviceID"]=="169"]
df.head(10)

e=df.groupby(["WeekOfYear","ProductID"], as_index=False).agg(
    QuantityProduced=("QuantityProduced","sum"),
    QuantityScrap=("QuantityScrap","sum"),
).sort_values(["WeekOfYear","ProductID"])
e["ScrapRate"]=e["QuantityScrap"]/e["QuantityProduced"]*100
print(e.head(30))

fig=px.box(
    e,
    x="ScrapRate",
    y="ProductID",
    title="Weekly Scrap Rate Distribution by Product on Device 169",
    labels={
        "ScrapRate": "Scrap Rate (%)",
        "ProductID": "Product"
    }
)

fig.update_layout(
    height=1000
)
fig.update_xaxes(ticksuffix="%", automargin=True)
fig.update_yaxes(automargin=True)

fig.show()


   WeekOfYear ProductID  QuantityProduced  QuantityScrap  ScrapRate
0     2023-22       332             117.0            0.0   0.000000
1     2023-22       443             401.0            0.0   0.000000
2     2023-23       189            3533.0            2.0   0.056609
3     2023-23       332            7621.0           21.0   0.275554
4     2023-24       227            3463.0           17.0   0.490904
5     2023-24       332           14164.0           16.0   0.112962
6     2023-25       302            2680.0            9.0   0.335821
7     2023-25       322            7121.0           54.0   0.758320
8     2023-25       332            1221.0            2.0   0.163800
9     2023-25       370            6011.0           34.0   0.565630
10    2023-26       297           10562.0          972.0   9.202802
11    2023-26       302            2256.0            3.0   0.132979
12    2023-27       297           10746.0          908.0   8.449656
13    2023-28       189            3544.0       

In [17]:
# Device 169 only
d169 = df[df["DeviceID"].astype(str) == "169"].copy()

# Recalculate scrap rate at Product × Week level
d169["ScrapRate"] = (
    d169["QuantityScrap"] / d169["QuantityProduced"] * 100
)

# Total production of Device 169 in each week
weekly_total = (
    d169.groupby("WeekOfYear")["QuantityProduced"]
        .transform("sum")
)

# Each product's share of Device 169 weekly production
d169["ProductionSharePct"] = (
    d169["QuantityProduced"] / weekly_total * 100
)

# Product 297 only
p297 = d169[d169["ProductID"].astype(str) == "297"].copy()



fig = px.bar(
    p297,
    x="WeekOfYear",
    y="ScrapRate",
    text_auto=".1f",
    title="Product 297 Scrap Rate on Device 169",
    labels={
        "WeekOfYear": "Week",
        "ScrapRate": "Scrap Rate (%)"
    }
)

fig.update_xaxes(tickangle=-45, automargin=True)
fig.update_yaxes(ticksuffix="%", automargin=True)
fig.show()


fig = px.bar(
    p297,
    x="WeekOfYear",
    y="ProductionSharePct",
    text_auto=".1f",
    title="Product 297 Share of Device 169 Weekly Production",
    labels={
        "WeekOfYear": "Week",
        "ProductionSharePct": "Share of Weekly Production (%)"
    }
)

fig.update_xaxes(tickangle=-45, automargin=True)
fig.update_yaxes(ticksuffix="%", automargin=True)
fig.show()

In [18]:


query=""" 
WITH t1 AS
(
SELECT ProductID,DeviceID,strftime('%Y-%W',StartTime) AS WeekOfYear,SUM(QuantityScrap) AS QuantityScrap,SUM(QuantityProduced) AS QuantityProduced,SUM((julianday(EndTime)-julianday(StartTime))*24) AS TotalReportedHours, SUM(((julianday(EndTime)-julianday(StartTime))*24)-(InteruptMinutes/60.0)) AS EstimatedRunHours
FROM scamp_report_3m
GROUP BY ProductID,DeviceID,strftime('%Y-%W',StartTime)
),

t2 AS
(
SELECT *, (QuantityProduced/TotalReportedHours) AS ObservedThroughput, (QuantityProduced/EstimatedRunHours) AS RunTimeThroughput
FROM t1

)

SELECT *,(RunTimeThroughput - ObservedThroughput)*100/RunTimeThroughput AS EstimatedThroughputLossPct
FROM t2



"""

df=run_query(query,conn)

df["DeviceID"]=df["DeviceID"].astype(str)
df["ProductID"]=df["ProductID"].astype(str)
df.head(10)

df=df[df["ProductID"]=="297"]

f=df.groupby(["WeekOfYear","DeviceID"], as_index=False).agg(
    QuantityProduced=("QuantityProduced","sum"),
    QuantityScrap=("QuantityScrap","sum"),
).sort_values(["WeekOfYear","DeviceID"])
f["ScrapRate"]=f["QuantityScrap"]/f["QuantityProduced"]*100


f.head()

fig=px.line(
    f,
    x="WeekOfYear",
    y="ScrapRate",
    color="DeviceID",
    title="Weekly Scrap Rate for Product 297 by Machine",
    labels={
        "WeekOfYear": "Week",
        "ScrapRate": "Scrap Rate (%)",
        "DeviceID": "Machine"
    }
)

fig.update_layout(
    height=1000
)
fig.update_xaxes(tickangle=-45, automargin=True)
fig.update_yaxes(ticksuffix="%", automargin=True)

fig.show()

### Summary

- Product 297 records high scrap on Device 169 in four consecutive weeks: about **9.20% in week 26, 8.45% in week 27, 6.15% in week 28, and 11.78% in week 29**.
- Product 297 is also a major share of Device 169 production in those weeks, so it materially affects the machine's overall scrap result.
- Across the full period, Product 297 scrap is about **7.63% on Device 169**, compared with approximately **0.94% on Device 168, 0.93% on Device 170, and 1.15% on Device 172**.

This is the strongest root-cause direction in the dataset. The same product performs much worse on Device 169, so the next investigation should compare tooling, machine settings, material batches, operator or shift, and quality notes for those runs.

## 8. Interruption around job changes

Each report is classified as the first report after a job change, the last report before a job change, or neither. The comparison checks whether recorded interruption time is systematically different around transitions.

In [19]:
query = """ 

WITH t1 AS
(
SELECT *,LAG(JobID) OVER (PARTITION BY DeviceID ORDER BY StartTime ) AS PreviousJobID
FROM scamp_report_3m
),

t2 AS
(
SELECT *,
    CASE WHEN PreviousJobID <> JobID THEN 1
    ELSE 0 END AS FirstAfterJobChange
FROM t1
),

t3 AS
(
SELECT *, LEAD(FirstAfterJobChange) OVER (PARTITION BY DeviceID ORDER BY StartTime) AS LastBeforeJobChange
FROM t2
)


SELECT ReportID,DeviceID,InteruptMinutes,
    CASE 
    WHEN FirstAfterJobChange = 1 THEN 'First After Job Change'
    WHEN LastBeforeJobChange = 1 THEN 'Last Before Job Change'
    ELSE 'Neither' END AS JobChangeStatus
FROM t3



"""
df=run_query(query,conn)
print(df.head())

   ReportID  DeviceID  InteruptMinutes         JobChangeStatus
0    291173       168         4.250000                 Neither
1    291201       168         0.000000  Last Before Job Change
2    291255       168        14.150000  First After Job Change
3    291314       168         9.933333                 Neither
4    291323       168         4.983333                 Neither


In [20]:
df["DeviceID"] = df["DeviceID"].astype(str)

fig = px.box(
    df,
    x="JobChangeStatus",
    y="InteruptMinutes",
    facet_col="DeviceID",
    facet_col_wrap=2,
    points="outliers",
    title="Interruption Duration Around Job Changes by Machine",
    labels={
        "JobChangeStatus": "Position Relative to Job Change",
        "InteruptMinutes": "Interruption (minutes)",
        "DeviceID": "Machine"
    }
)

fig.update_layout(height=1000)
fig.update_xaxes(tickangle=-25, automargin=True)
fig.update_yaxes(automargin=True)
fig.show()

### Summary

The clearest pattern occurs in the **first report after a job change**. Median interruption minutes are higher than normal on every machine:

| Device | First after change | Neither |
|---:|---:|---:|
| 168 | 34.22 min | 2.83 min |
| 169 | 20.26 min | 5.10 min |
| 170 | 17.33 min | 2.15 min |
| 172 | 14.37 min | 6.63 min |
| 321 | 18.67 min | 2.05 min |

The last report before a change is generally much closer to normal production. This suggests that restart, setup confirmation, first-piece inspection, or early-run adjustment may be recorded inside the first report of the new job. The dataset does not contain interruption reasons, so this remains a strong investigation direction rather than a confirmed cause.

## 9. Conclusions and recommended next steps

### Main conclusions

1. Production reports do not overlap within a machine, and the usual short gap is around eight minutes. Most gaps occur without a job change.
2. Interruption performance depends on both product and machine. High-percentage cases with very little run time need to be separated from repeated, high-exposure losses.
3. Device 169 is the main scrap concern. Product 297 explains a large part of the problem and performs far worse there than on the other machines.
4. The first report after a job change has consistently higher interruption time across all five machines, making post-change startup activity an important improvement area.

### Recommended next steps

- Review Product 297 runs on Device 169 during weeks 26–29 and compare settings, tooling condition, material batch, operator or shift, and defect type.
- Break `InteruptMinutes` into reason categories if a downtime or operator log is available. Without reason codes, planned setup and unplanned loss cannot be separated reliably.
- Track a job-transition KPI for the first report after each change and compare it with normal within-job production.
- Apply minimum reported-hour or production-volume thresholds before ranking product–machine combinations.
- Exclude partial weeks, or clearly mark them, when weekly machine performance is used for targets.

### Limitations

- The analysis is observational and covers only about three months.
- No shift, operator, material batch, defect category, maintenance event, or interruption reason is available.
- Estimated runtime assumes the interruption field is measured consistently and can be subtracted from report duration.
- The findings identify where to investigate; they do not by themselves prove why a machine or product performed differently.